# Lily's Fast Video Studio — Kaggle
**Import this notebook, select GPU T4 x2, switch Internet ON, and run the three cells in order.**

1. **Set up** the isolated runtime. This leaves Kaggle's installed Python packages alone.
2. **Download** the small LTX-Video 0.9.8 2B Distilled model and text encoder. Progress is shown here; completed files are reused within the session.
3. **Open** the studio. It loads the models, runs a tiny startup render, and prints a **gradio.live** link. Open that link on your iPhone.

Start with **Balanced · 5 seconds**. A photo is optional. Choose Fast for quicker previews or More detail for a larger render. There is no audio.

The first setup includes roughly **16 GB of model downloads**, plus the Python runtime. Later clips reuse the loaded models. T4 x2 uses one card for video and the other for prompt encoding; it does not pretend the cards form one 32 GB GPU. One 16 GB GPU can run with model swapping, more slowly.

Generation speed and quality have not been benchmarked on your Kaggle GPU. The startup test checks the actual session before declaring it ready. Longer and larger clips take more time; a 10-second clip can cost more than twice a 5-second clip. Kaggle session limits still apply.

Keep the last cell running while using the studio. Save videos from its Download MP4 button before ending the session. Only MP4s and settings go into `/kaggle/working/lily_videos`; model files stay in temporary storage. Downloads may be needed again in a new session.


In [ ]:
# 1 — Set up (first run installs the environment)
import os, sys, subprocess
from pathlib import Path
BASE = Path('/kaggle/temp/lily-fast-v3')
BASE.mkdir(parents=True, exist_ok=True)
FILES = {'download_models.py': "import os\nos.environ['HF_HUB_DISABLE_XET'] = '1'\nos.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'\nos.environ['HF_HUB_ETAG_TIMEOUT'] = '60'\nimport json, shutil, time\nfrom pathlib import Path\nfrom huggingface_hub import hf_hub_download, snapshot_download\nfrom safetensors import safe_open\nBASE = Path(os.environ.get('LILY_BASE', '/kaggle/temp/lily-fast-v3'))\nBASE.mkdir(parents=True, exist_ok=True)\nCACHE = BASE / 'cache'\nMODEL_REPO = 'Lightricks/LTX-Video'\nMODEL_REV = '8984fa25007f376c1a299016d0957a37a2f797bb'\nTEXT_REPO = 'PixArt-alpha/PixArt-XL-2-1024-MS'\nTEXT_REV = 'b89adadeccd9ead2adcb9fa2825d3fabec48d404'\nCHECKPOINT = 'ltxv-2b-0.9.8-distilled.safetensors'\n\n\ndef retry(fn):\n    for attempt in range(3):\n        try:\n            return fn()\n        except Exception as exc:\n            if attempt == 2:\n                raise RuntimeError('Download did not finish. Check Kaggle Internet is ON and rerun this cell; completed files are reused.') from exc\n            print(f'Download interrupted ({type(exc).__name__}); resuming attempt {attempt+2}/3...', flush=True)\n            time.sleep(2)\n\n\ndef main():\n    free = shutil.disk_usage(BASE).free / 1024**3\n    print(f'Model cache: {BASE}. Free space: {free:.1f} GiB.', flush=True)\n    if free < 20 and not (BASE / 'downloads.json').exists():\n        raise RuntimeError('Need at least 20 GiB free for the models. Start a fresh Kaggle session.')\n    print('1/2 Downloading/checking LTX 2B (about 6.3 GB)...', flush=True)\n    checkpoint = retry(lambda: hf_hub_download(MODEL_REPO, CHECKPOINT, revision=MODEL_REV, cache_dir=str(CACHE)))\n    with safe_open(checkpoint, framework='pt', device='cpu') as f:\n        metadata = json.loads(f.metadata()['config'])\n        if metadata['transformer']['num_layers'] != 28:\n            raise RuntimeError('Unexpected video checkpoint architecture.')\n    print('2/2 Downloading/checking the text encoder and tokenizer (about 9.5 GB)...', flush=True)\n    text_path = retry(lambda: snapshot_download(TEXT_REPO, revision=TEXT_REV, cache_dir=str(CACHE),\n        allow_patterns=['text_encoder/*.safetensors', 'text_encoder/*.json', 'tokenizer/*'], max_workers=3))\n    if not list((Path(text_path) / 'text_encoder').glob('*.safetensors')):\n        raise RuntimeError('The text encoder weights are missing; rerun this cell.')\n    manifest = {'checkpoint': checkpoint, 'text': text_path, 'model_revision': MODEL_REV, 'text_revision': TEXT_REV}\n    (BASE / 'downloads.json').write_text(json.dumps(manifest, indent=2))\n    print('DOWNLOADS COMPLETE. Run the next cell to open your studio.', flush=True)\n\nif __name__ == '__main__':\n    main()\n", 'lily_studio.py': '"""Lily\'s Kaggle video studio. Embedded in the importable notebook."""\nimport os\nos.environ.setdefault(\'HF_HUB_OFFLINE\', \'1\')\nos.environ.setdefault(\'TRANSFORMERS_OFFLINE\', \'1\')\nos.environ.setdefault(\'TOKENIZERS_PARALLELISM\', \'false\')\nos.environ.setdefault(\'GRADIO_ANALYTICS_ENABLED\', \'false\')\nimport gc, json, time, secrets, traceback, threading\nfrom collections import OrderedDict\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport gradio as gr\nimport imageio.v2 as imageio\nfrom PIL import Image, ImageOps\nfrom transformers import T5EncoderModel, T5Tokenizer\nfrom diffusers import AutoencoderKLLTXVideo, LTXVideoTransformer3DModel\nfrom diffusers import LTXConditionPipeline, FlowMatchEulerDiscreteScheduler\n\nBASE = Path(os.environ.get(\'LILY_BASE\', \'/kaggle/temp/lily-fast-v3\'))\nOUTPUT = Path(os.environ.get(\'LILY_OUTPUT\', \'/kaggle/working/lily_videos\'))\nTIMESTEPS = [1000.0, 993.7, 987.5, 981.2, 975.0, 909.4, 725.0]\nPRESETS = {\'Fast\': (512, 320), \'Balanced\': (640, 384), \'More detail\': (768, 448)}\nFPS = 24\nLOCK = threading.Lock()\n\n\ndef video_settings(preset, orientation, seconds):\n    if preset not in PRESETS or orientation not in (\'Landscape\', \'Portrait\', \'Square\'):\n        raise ValueError(\'Choose a valid quality and shape.\')\n    seconds = int(seconds)\n    if seconds not in (5, 10):\n        raise ValueError(\'Choose 5 or 10 seconds.\')\n    width, height = PRESETS[preset]\n    if orientation == \'Portrait\':\n        width, height = height, width\n    elif orientation == \'Square\':\n        width = height = int((width * height) ** 0.5) // 32 * 32\n    frames = seconds * FPS + 1  # LTX requires 8n+1 frames; 121 or 241.\n    return width, height, frames\n\n\ndef export_mp4(frames, path):\n    array = np.asarray(frames)\n    if array.ndim != 4 or array.shape[-1] != 3 or not np.isfinite(array).all():\n        raise RuntimeError(\'Invalid video frames; no broken MP4 was saved.\')\n    with imageio.get_writer(str(path), fps=FPS, codec=\'libx264\', quality=8,\n                           macro_block_size=None, pixelformat=\'yuv420p\',\n                           ffmpeg_params=[\'-movflags\', \'+faststart\']) as writer:\n        for frame in array:\n            writer.append_data((np.clip(frame, 0, 1) * 255).round().astype(np.uint8))\n    if not path.is_file() or path.stat().st_size < 500:\n        raise RuntimeError(\'The video encoder did not produce a valid file.\')\n\n\nclass Studio:\n    def __init__(self):\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'In Kaggle Settings, select GPU T4 x2 and start a new session.\')\n        self.device = torch.device(\'cuda:0\')\n        self.dual = torch.cuda.device_count() >= 2\n        self.text_device = torch.device(\'cuda:1\' if self.dual else \'cuda:0\')\n        self.dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16\n        self.text_dtype = torch.bfloat16 if torch.cuda.get_device_capability(self.text_device)[0] >= 8 else torch.float16\n        self.cache = OrderedDict()\n        self.compatibility = False\n        torch.set_grad_enabled(False)\n        torch.backends.cuda.matmul.allow_tf32 = self.dtype == torch.bfloat16\n        torch.backends.cudnn.allow_tf32 = self.dtype == torch.bfloat16\n        # A real kernel test catches incompatible CUDA builds before the large models load.\n        for index in range(torch.cuda.device_count()):\n            dev = torch.device(f\'cuda:{index}\')\n            sample = torch.ones((32, 32), device=dev, dtype=torch.float16)\n            _ = (sample @ sample).sum().item()\n            print(f\'GPU {index}: {torch.cuda.get_device_name(index)}\', flush=True)\n        del sample\n        manifest = json.loads((BASE / \'downloads.json\').read_text())\n        checkpoint = manifest[\'checkpoint\']\n        configs = BASE / \'configs\'\n        print(\'Loading the small distilled video model...\', flush=True)\n        transformer = LTXVideoTransformer3DModel.from_single_file(\n            checkpoint, config=str(configs), subfolder=\'transformer\',\n            torch_dtype=self.dtype, local_files_only=True).eval().to(self.device)\n        vae = AutoencoderKLLTXVideo.from_single_file(\n            checkpoint, config=str(configs), subfolder=\'vae\',\n            torch_dtype=self.dtype, local_files_only=True).eval().to(self.device)\n        vae.enable_tiling(tile_sample_min_height=256, tile_sample_min_width=256,\n                          tile_sample_min_num_frames=32, tile_sample_stride_height=192,\n                          tile_sample_stride_width=192, tile_sample_stride_num_frames=24)\n        self.tokenizer = T5Tokenizer.from_pretrained(manifest[\'text\'], subfolder=\'tokenizer\', local_files_only=True)\n        print(\'Loading the prompt encoder...\', flush=True)\n        self.encoder = T5EncoderModel.from_pretrained(\n            manifest[\'text\'], subfolder=\'text_encoder\', torch_dtype=self.text_dtype,\n            low_cpu_mem_usage=True, use_safetensors=True, local_files_only=True).eval()\n        if self.dual:\n            self.encoder.to(self.text_device)\n        self.pipe = LTXConditionPipeline(\n            transformer=transformer, vae=vae, text_encoder=None, tokenizer=self.tokenizer,\n            scheduler=FlowMatchEulerDiscreteScheduler(shift=1.0, use_dynamic_shifting=False))\n        self.pipe.set_progress_bar_config(disable=False)\n        OUTPUT.mkdir(parents=True, exist_ok=True)\n        gc.collect()\n        torch.cuda.empty_cache()\n        print(\'All model downloads and loading are finished.\', flush=True)\n\n    @torch.inference_mode()\n    def encode(self, prompt):\n        if prompt in self.cache:\n            self.cache.move_to_end(prompt)\n            return self.cache[prompt]\n        print(\'Encoding your prompt...\', flush=True)\n        tokens = self.tokenizer(prompt.strip(), padding=\'max_length\', max_length=256,\n                                truncation=True, return_tensors=\'pt\')\n        if not self.dual:\n            # The 16 GB cards do not share VRAM. Make room for T5 on a single card.\n            self.pipe.transformer.to(\'cpu\')\n            self.pipe.vae.to(\'cpu\')\n            gc.collect()\n            torch.cuda.empty_cache()\n        try:\n            self.encoder.to(self.text_device)\n            inputs = {k: v.to(self.text_device) for k, v in tokens.items()}\n            embeddings = self.encoder(**inputs).last_hidden_state\n            if not torch.isfinite(embeddings).all():\n                raise FloatingPointError(\'Prompt encoder overflow in half precision.\')\n            result = embeddings.cpu(), tokens.attention_mask.cpu()\n        except FloatingPointError:\n            print(\'Retrying this prompt in full precision on CPU...\', flush=True)\n            self.encoder.to(\'cpu\', dtype=torch.float32)\n            gc.collect()\n            torch.cuda.empty_cache()\n            embeddings = self.encoder(**tokens).last_hidden_state\n            if not torch.isfinite(embeddings).all():\n                raise RuntimeError(\'Prompt encoding produced non-finite values.\')\n            # Scale only if needed to fit FP16; do not silently cast large values to infinity.\n            if embeddings.abs().max().item() > 65000:\n                raise RuntimeError(\'Prompt embeddings exceed FP16 range; try a shorter, simpler prompt.\')\n            result = embeddings.cpu(), tokens.attention_mask.cpu()\n            self.encoder.to(dtype=self.text_dtype)\n        finally:\n            if not self.dual:\n                self.encoder.to(\'cpu\')\n                gc.collect()\n                torch.cuda.empty_cache()\n                self.pipe.transformer.to(self.device)\n                self.pipe.vae.to(self.device)\n        self.cache[prompt] = result\n        while len(self.cache) > 8:\n            self.cache.popitem(last=False)\n        return result\n\n    @torch.inference_mode()\n    def render(self, prompt, image, width, height, frames, seed, progress=None):\n        embeddings, mask = self.encode(prompt)\n        embeddings = embeddings.to(self.device, dtype=self.pipe.transformer.dtype)\n        mask = mask.to(self.device)\n        if not torch.isfinite(embeddings).all():\n            raise FloatingPointError(\'Prompt embeddings overflowed on transfer.\')\n        if image is not None:\n            image = ImageOps.fit(ImageOps.exif_transpose(image).convert(\'RGB\'),\n                                 (width, height), method=Image.Resampling.LANCZOS)\n        def on_step(pipe, step, timestep, kwargs):\n            if not torch.isfinite(kwargs[\'latents\']).all():\n                raise FloatingPointError(\'Half-precision video computation overflowed.\')\n            if progress:\n                progress((step + 1) / (len(TIMESTEPS) + 2),\n                         desc=f\'Rendering step {step + 1}/{len(TIMESTEPS)}\')\n            return kwargs\n        print(f\'Rendering {width}x{height}, {frames} frames, {len(TIMESTEPS)} steps...\', flush=True)\n        torch.manual_seed(seed)  # Also seeds the VAE decode noise.\n        result = self.pipe(\n            image=image, prompt=None, prompt_embeds=embeddings, prompt_attention_mask=mask,\n            width=width, height=height, num_frames=frames, frame_rate=FPS,\n            timesteps=TIMESTEPS, guidance_scale=1.0, guidance_rescale=0.0,\n            image_cond_noise_scale=0.0, decode_timestep=0.05, decode_noise_scale=0.025,\n            generator=torch.Generator(device=self.device).manual_seed(seed),\n            output_type=\'np\', callback_on_step_end=on_step,\n            callback_on_step_end_tensor_inputs=[\'latents\']).frames[0]\n        if result.shape != (frames, height, width, 3) or not np.isfinite(result).all():\n            raise FloatingPointError(\'The video decoder produced invalid frames.\')\n        return result\n\n    def enable_full_precision(self):\n        # Compatibility fallback for older cards, at a speed and memory cost.\n        if self.compatibility:\n            raise RuntimeError(\'Generation also failed in full precision. Restart the session and use Fast.\')\n        print(\'Switching video computation to full precision for this older GPU.\', flush=True)\n        self.pipe.transformer.to(\'cpu\')\n        self.pipe.vae.to(\'cpu\')\n        gc.collect()\n        torch.cuda.empty_cache()\n        self.pipe.transformer.to(dtype=torch.float32, device=self.device)\n        self.pipe.vae.to(dtype=torch.float32, device=self.device)\n        self.compatibility = True\n\n    def startup_check(self):\n        prompt = \'A green tree moves gently in the breeze in soft daylight. The camera is still.\'\n        start = time.perf_counter()\n        failure = None\n        try:\n            frames = self.render(prompt, None, 256, 160, 17, 42)\n        except FloatingPointError as exc:\n            failure = str(exc)\n        if failure:\n            gc.collect()\n            torch.cuda.empty_cache()\n            self.enable_full_precision()\n            frames = self.render(prompt, None, 256, 160, 17, 42)\n        export_mp4(frames, OUTPUT / \'startup_check.mp4\')\n        print(f\'Startup render and MP4 check passed ({time.perf_counter() - start:.1f}s).\', flush=True)\n        del frames\n\n    def generate(self, prompt, image, preset, orientation, seconds, seed, progress=gr.Progress()):\n        if not prompt or not prompt.strip():\n            raise gr.Error(\'Write what happens in the video first.\')\n        width, height, frames = video_settings(preset, orientation, seconds)\n        seed = int(seed) if seed is not None else -1\n        if seed < 0:\n            seed = secrets.randbelow(2**31)\n        if not 0 <= seed < 2**32:\n            raise gr.Error(\'Seed must be -1 (random) or an integer from 0 to 4294967295.\')\n        start = time.perf_counter()\n        notes = []\n        with LOCK:\n            for attempt in range(3):\n                problem = None\n                try:\n                    progress(0, desc=\'Preparing your prompt\')\n                    output = self.render(prompt, image, width, height, frames, seed, progress)\n                    break\n                except torch.cuda.OutOfMemoryError:\n                    problem = \'memory\'\n                except FloatingPointError:\n                    problem = \'precision\'\n                # Retry after leaving the exception handler so failed tensors can be freed.\n                gc.collect()\n                torch.cuda.empty_cache()\n                if problem == \'precision\' and not self.compatibility:\n                    self.enable_full_precision()\n                    notes.append(\'Full-precision compatibility mode\')\n                elif problem == \'memory\' and attempt < 2:\n                    width = max(128, int(width * 0.75) // 32 * 32)\n                    height = max(128, int(height * 0.75) // 32 * 32)\n                    notes.append(f\'Memory retry at {width}x{height}\')\n                    gr.Info(f\'Memory was tight. Retrying at {width}x{height}.\')\n                else:\n                    raise gr.Error(\'This GPU could not finish. Choose Fast and 5 seconds, or restart the session.\')\n            else:\n                raise gr.Error(\'Generation did not complete. Restart the session and try Fast.\')\n            progress(0.93, desc=\'Saving your video\')\n            path = OUTPUT / f\'lily_{time.time_ns()}_{seed}.mp4\'\n            export_mp4(output, path)\n            elapsed = time.perf_counter() - start\n            record = dict(prompt=prompt, seed=seed, width=width, height=height, frames=frames,\n                          fps=FPS, seconds=frames/FPS, elapsed_seconds=elapsed, notes=notes,\n                          model=\'LTX-Video 0.9.8 2B Distilled\', image_conditioned=image is not None)\n            path.with_suffix(\'.json\').write_text(json.dumps(record, indent=2))\n            del output\n            gc.collect()\n            torch.cuda.empty_cache()\n        summary = f\'{frames/FPS:.2f}s video · {width} × {height} · made in {elapsed:.1f}s · seed {seed}\'\n        if notes:\n            summary += \'\\n\\n\' + \'; \'.join(notes)\n        progress(1, desc=\'Done\')\n        return str(path), str(path), summary\n\n\ndef build_ui(studio):\n    with gr.Blocks(title="Lily\'s Fast Video Studio", theme=gr.themes.Soft()) as demo:\n        gr.Markdown("# Lily\'s Video Studio\\nUpload a photo or start with words. Describe one clear motion for the best results.")\n        with gr.Row():\n            with gr.Column():\n                reference = gr.Image(type=\'pil\', sources=[\'upload\'], label=\'Starting photo (optional)\')\n                prompt = gr.Textbox(label=\'What happens?\', lines=4,\n                    placeholder=\'The person smiles and turns toward the camera. A light breeze moves their hair. Soft evening light.\')\n                preset = gr.Radio(list(PRESETS), value=\'Balanced\', label=\'Quality\')\n                shape = gr.Radio([\'Landscape\', \'Portrait\', \'Square\'], value=\'Portrait\', label=\'Shape\')\n                seconds = gr.Radio([5, 10], value=5, label=\'Seconds\')\n                with gr.Accordion(\'Seed\', open=False):\n                    seed = gr.Number(value=-1, precision=0, label=\'-1 = a new variation each time\')\n                button = gr.Button(\'Make my video\', variant=\'primary\')\n            with gr.Column():\n                video = gr.Video(label=\'Your video\', format=\'mp4\')\n                download = gr.File(label=\'Download MP4\')\n                status = gr.Markdown(\'Ready. Start with Balanced and 5 seconds.\')\n        button.click(studio.generate, [prompt, reference, preset, shape, seconds, seed],\n                     [video, download, status], concurrency_limit=1, api_name=False)\n        gr.Markdown(\'More detail and 10-second clips take longer. Keep the Kaggle session running. Videos have no audio.\')\n    return demo\n\n\ndef main():\n    studio = Studio()\n    studio.startup_check()\n    demo = build_ui(studio)\n    demo.queue(default_concurrency_limit=1, max_size=4)\n    print(\'Opening the studio. Use the gradio.live link below on your phone.\', flush=True)\n    demo.launch(share=True, server_name=\'0.0.0.0\', show_error=True, inbrowser=False,\n                allowed_paths=[str(OUTPUT)], max_file_size=\'30mb\')\n\n\nif __name__ == \'__main__\':\n    try:\n        main()\n    except Exception:\n        traceback.print_exc()\n        raise\n', 'setup_environment.py': 'import os, sys, subprocess, json, hashlib, shutil\nfrom pathlib import Path\nBASE = Path(\'/kaggle/temp/lily-fast-v3\')\nBASE.mkdir(parents=True, exist_ok=True)\nos.environ[\'LILY_BASE\'] = str(BASE)\nos.environ[\'LILY_OUTPUT\'] = \'/kaggle/working/lily_videos\'\nos.environ[\'PIP_CACHE_DIR\'] = str(BASE / \'pip-cache\')\nos.environ[\'HF_HOME\'] = str(BASE / \'cache\')\nos.environ[\'GRADIO_TEMP_DIR\'] = str(BASE / \'uploads\')\nos.environ[\'TOKENIZERS_PARALLELISM\'] = \'false\'\nos.environ[\'PYTHONUNBUFFERED\'] = \'1\'\nPYTHON = BASE / \'venv/bin/python\'\nif not (3, 10) <= sys.version_info[:2] <= (3, 12):\n    raise RuntimeError(\'This notebook requires Python 3.10–3.12, the supported range for its pinned GPU runtime.\')\ntry:\n    subprocess.run([\'nvidia-smi\', \'--query-gpu=name,memory.total\', \'--format=csv,noheader\'], check=True)\nexcept (FileNotFoundError, subprocess.CalledProcessError) as exc:\n    raise RuntimeError(\'Set Kaggle Settings → Accelerator → GPU T4 x2, then start a new session.\') from exc\n\ndef run(command):\n    subprocess.run([str(part) for part in command], check=True)\n\nif not PYTHON.exists():\n    if shutil.disk_usage(BASE).free < 30 * 1024**3:\n        raise RuntimeError(\'Please start a fresh Kaggle session with at least 30 GiB free.\')\n    run([sys.executable, \'-m\', \'venv\', str(BASE / \'venv\')])\nrequirements = (BASE / \'requirements.txt\').read_text()\nsignature = hashlib.sha256((\'torch==2.5.1+cu121\\ntorchvision==0.20.1+cu121\\n\' + requirements).encode()).hexdigest()\nmarker = BASE / \'environment-ready.txt\'\nif not marker.exists() or marker.read_text() != signature:\n    print(\'Installing the pinned GPU runtime in an isolated environment. First run only.\', flush=True)\n    run([PYTHON, \'-m\', \'pip\', \'install\', \'--disable-pip-version-check\', \'--no-cache-dir\',\n         \'torch==2.5.1\', \'torchvision==0.20.1\', \'--index-url\', \'https://download.pytorch.org/whl/cu121\'])\n    run([PYTHON, \'-m\', \'pip\', \'install\', \'--disable-pip-version-check\', \'--no-cache-dir\',\n         \'-r\', BASE / \'requirements.txt\'])\n    run([PYTHON, \'-m\', \'pip\', \'check\'])\n    marker.write_text(signature)\nrun([PYTHON, \'-c\', "import torch, gradio; from diffusers import LTXConditionPipeline; assert torch.cuda.is_available(), \'GPU unavailable\'; x=torch.ones((32,32),device=\'cuda\',dtype=torch.float16); print(\'CUDA check:\',(x@x).sum().item()); print(\'Runtime ready. Run the download cell next.\')"])\n', 'requirements.txt': 'diffusers==0.35.2\ntransformers==4.51.3\nhuggingface-hub==0.35.3\naccelerate==1.6.0\nsafetensors==0.5.3\nsentencepiece==0.2.0\ngradio==5.49.1\nimageio==2.37.0\nimageio-ffmpeg==0.6.0\nnumpy==1.26.4\npillow==11.2.1\nbeautifulsoup4==4.13.4\nftfy==6.3.1\n\nhttpx[socks]==0.28.1\n', 'configs/vae/config.json': '{"_class_name": "AutoencoderKLLTXVideo", "_diffusers_version": "0.33.0.dev0", "block_out_channels": [128, 256, 512, 1024, 2048], "decoder_block_out_channels": [256, 512, 1024], "decoder_causal": false, "decoder_inject_noise": [false, false, false, false], "decoder_layers_per_block": [5, 5, 5, 5], "decoder_spatio_temporal_scaling": [true, true, true], "down_block_types": ["LTXVideo095DownBlock3D", "LTXVideo095DownBlock3D", "LTXVideo095DownBlock3D", "LTXVideo095DownBlock3D"], "downsample_type": ["spatial", "temporal", "spatiotemporal", "spatiotemporal"], "encoder_causal": true, "in_channels": 3, "latent_channels": 128, "layers_per_block": [4, 6, 6, 2, 2], "out_channels": 3, "patch_size": 4, "patch_size_t": 1, "resnet_norm_eps": 1e-06, "scaling_factor": 1.0, "spatial_compression_ratio": 32, "spatio_temporal_scaling": [true, true, true, true], "temporal_compression_ratio": 8, "timestep_conditioning": true, "upsample_factor": [2, 2, 2], "upsample_residual": [true, true, true]}', 'configs/transformer/config.json': '{"_class_name": "LTXVideoTransformer3DModel", "_diffusers_version": "0.33.0.dev0", "activation_fn": "gelu-approximate", "attention_bias": true, "attention_head_dim": 64, "attention_out_bias": true, "caption_channels": 4096, "cross_attention_dim": 2048, "in_channels": 128, "norm_elementwise_affine": false, "norm_eps": 1e-06, "num_attention_heads": 32, "num_layers": 28, "out_channels": 128, "patch_size": 1, "patch_size_t": 1, "qk_norm": "rms_norm_across_heads"}'}
for relative, contents in FILES.items():
    destination = BASE / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(contents)
exec(compile(FILES['setup_environment.py'], str(BASE / 'setup_environment.py'), 'exec'))


In [ ]:
# 2 — Download all models with progress (rerunnable)
import os, subprocess
from pathlib import Path
BASE = Path('/kaggle/temp/lily-fast-v3')
PYTHON = BASE / 'venv/bin/python'
assert PYTHON.exists(), 'Run cell 1 first.'
subprocess.run([str(PYTHON), '-u', str(BASE / 'download_models.py')], check=True)


In [ ]:
# 3 — Load, test, and open the studio. Keep this cell running.
import os, subprocess
from pathlib import Path
BASE = Path('/kaggle/temp/lily-fast-v3')
PYTHON = BASE / 'venv/bin/python'
assert (BASE / 'downloads.json').exists(), 'Run cell 2 first.'
# Stop only the earlier studio launched by this notebook if this cell is rerun.
previous = globals().get('lily_process')
if previous is not None and previous.poll() is None:
    previous.terminate()
    try:
        previous.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous.kill()
        previous.wait()
lily_process = subprocess.Popen([str(PYTHON), '-u', str(BASE / 'lily_studio.py')])
try:
    code = lily_process.wait()
    if code:
        raise RuntimeError('Studio stopped. The error is printed above; please share that output.')
except KeyboardInterrupt:
    lily_process.terminate()
    try:
        lily_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        lily_process.kill()
        lily_process.wait()
    print('Studio stopped. Run this cell again to reopen it.')


### Sources and validation
- [Official LTX model files](https://huggingface.co/Lightricks/LTX-Video/tree/8984fa25007f376c1a299016d0957a37a2f797bb)
- [Official 2B distilled sampling configuration](https://github.com/Lightricks/LTX-Video/blob/4b2d053057623ddd4d0a1d3e9cd28890e9ef487f/configs/ltxv-2b-0.9.8-distilled.yaml)
- [Hugging Face LTX pipeline documentation](https://huggingface.co/docs/diffusers/v0.35.1/en/api/pipelines/ltx_video)

The original 0.9.8 2B checkpoint is converted in memory by Diffusers using the matching 0.9.5 architecture configs. Every converted parameter name and tensor shape was checked against the actual checkpoint header. No community fine-tune, separate enhancement LLM, or custom compiled attention extension is required.

The model and text encoder revisions and principal Python dependencies are pinned. Local CPU tests cover model conversion structure, a tiny image-conditioned pipeline, the UI, and MP4 encoding. Full-size CUDA inference must be verified by the startup check on Kaggle; no universal error-free or speed guarantee is claimed.
